In [1]:
!pip install langchain-huggingface

In [2]:
! pip install datasets chromadb sentence-transformers langchain langchain-community langchain-text-splitters rank-bm25
! pip install torch transformers accelerate bitsandbytes huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB

In [3]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import numpy as np
import re
from typing import List, Dict
import torch

In [4]:
import transformers
transformers.logging.set_verbosity_error()

# transformers.logging.set_verbosity_error() is a Hugging Face Transformers utility function that sets the logging level to ERROR,
# meaning only error messages will be shown — warnings, info, and debug logs will be suppressed.

In [5]:
print("Loading neural-bridge/rag-dataset-1200 ...")
dataset = load_dataset("neural-bridge/rag-dataset-1200", split="train")

print(f"Dataset loaded  : {len(dataset)} examples")
print(f"Features        : {list(dataset.features.keys())}")

Loading neural-bridge/rag-dataset-1200 ...


README.md:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

data/train-00000-of-00001-f0c158413defd4(…): reconstructing file:   0%|          |  0.00B / 2.32MB            

data/train-00000-of-00001-f0c158413defd4(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-06d83c58a8ea10e(…): reconstructing file:   0%|          |  0.00B /  604kB            

data/test-00000-of-00001-06d83c58a8ea10e(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/960 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/240 [00:00<?, ? examples/s]

Dataset loaded  : 960 examples
Features        : ['context', 'question', 'answer']


In [6]:
print("\n \nSample entry:")
sample = dataset[0]
for key, val in sample.items():
    display_val = str(val)[:200] + "..." if len(str(val)) > 200 else str(val)
    print(f"  [{key}] {display_val}")


 
Sample entry:
  [context] Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essa...
  [question] Who found the answer to a search query collar george herbert essay?
  [answer] Francisco Rogers found the answer to a search query collar george herbert essay.


In [20]:
type(dataset)

datasets.arrow_dataset.Dataset

In [21]:
type(dataset[0])

dict

In [23]:
dataset[0:2]

{'context': ['Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society\nbuy apa format essay buy apa format essay\nconan doyle speckled band essay\ncollege essay application prompt\ncolumbia university mfa creative writing acceptance rate\ncrucible coursework questions\ncollege essay topics texas\ncover letter thesis proposal\nciting ma thesis\ncompare and contrast essays for elementary\ncoursework completed without degree\ncomparison islam christianity essay\ncheerleading stereotypes essay\ncultural diversity college essay\ncri

In [14]:
# dataset[0:5]["context"]

In [15]:
raw_docs = [
    Document(
        page_content=context,
        metadata={"source": f"doc_{index}", "doc_id": index}
    )
    for index, context in enumerate(dataset["context"])
]

len(raw_docs)

960

In [16]:
raw_docs[0]

Document(metadata={'source': 'doc_0', 'doc_id': 0}, page_content='Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society\nbuy apa format essay buy apa format essay\nconan doyle speckled band essay\ncollege essay application prompt\ncolumbia university mfa creative writing acceptance rate\ncrucible coursework questions\ncollege essay topics texas\ncover letter thesis proposal\nciting ma thesis\ncompare and contrast essays for elementary\ncoursework completed without degree\ncomparison islam christianity essay\ncheerleading ster

In [24]:
len(raw_docs[0].page_content)

2438

In [27]:
demo_raw_text = raw_docs[:100]
print(demo_raw_text[99])

page_content='Ah, another of my mo irregularly scheduled posts - my 'weekly' favourites! I have been going crazy for pink nude shades lately, on my face as well as my lips, so this week's favourites reflect that.
Fist up is Essie nail polish in muchi muchi, a cute pale pink shade that I've loved wearing to work lately. I don't own too many Essie polishes due to my undying love for Barry M, but I love this colour and the bottle is already almost half empty.
Next is a lipgloss in the form of Tanya Burr's Aurora, which is a gorgeous dusky pink that adds a nice natural rosy shine to my lips. The applicator on these lip glosses is the perfect shape to coat your lips in, I own two other shades in the range and they are definitely my new favourites!
Another lip product that's made it's way into my favourites is my new Maybelline ColorSensation lipstick in Tantalising Taupe, from the Stripped Nudes collection. The formula of these lipsticks are, in my opinion, superior to most other drugstore 

In [30]:
raw_docs = demo_raw_text

In [31]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators= ["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(raw_docs)
len(chunks)

901

In [32]:
chunk_lengths = [len(c.page_content) for c in chunks]

In [33]:
print(f"Original documents : {len(raw_docs)}")
print(f"Chunks produced    : {len(chunks)}")
print(f"Avg chunk size     : {np.mean(chunk_lengths):.0f} chars")
print(f"Min / Max          : {min(chunk_lengths)} / {max(chunk_lengths)} chars")

print("Example chunk:")
print("-" * 60)
print(chunks[100])

Original documents : 100
Chunks produced    : 901
Avg chunk size     : 365 chars
Min / Max          : 3 / 512 chars
Example chunk:
------------------------------------------------------------
page_content='- Use the REGISTER NODE command or the UPDATE NODE command to change the value of the SESSIONINITIATION parameter to SERVERONLY, Specify the high-level address and low-level address options. These options must match what the client is using, otherwise the server will not know how to contact the client.
- Set the scheduling mode to server-prompted. All sessions must be started by server-prompted scheduling on the port that was defined for the client with the REGISTER NODE or the UPDATE NODE commands.' metadata={'source': 'doc_13', 'doc_id': 13}


In [34]:
print(chunks[100].page_content)

- Use the REGISTER NODE command or the UPDATE NODE command to change the value of the SESSIONINITIATION parameter to SERVERONLY, Specify the high-level address and low-level address options. These options must match what the client is using, otherwise the server will not know how to contact the client.
- Set the scheduling mode to server-prompted. All sessions must be started by server-prompted scheduling on the port that was defined for the client with the REGISTER NODE or the UPDATE NODE commands.


In [35]:
chunks[100]

Document(metadata={'source': 'doc_13', 'doc_id': 13}, page_content='- Use the REGISTER NODE command or the UPDATE NODE command to change the value of the SESSIONINITIATION parameter to SERVERONLY, Specify the high-level address and low-level address options. These options must match what the client is using, otherwise the server will not know how to contact the client.\n- Set the scheduling mode to server-prompted. All sessions must be started by server-prompted scheduling on the port that was defined for the client with the REGISTER NODE or the UPDATE NODE commands.')

In [36]:
print("Loading Qwen/Qwen3-Embedding-0.6B ...")
embed_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Embedding model loaded")

Loading Qwen/Qwen3-Embedding-0.6B ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model loaded


In [37]:
# Sanity check — use encode_query() which applies the model's built-in query prompt
test_emb = embed_model.encode_query(
    "What is retrieval augmented generation?",
    normalize_embeddings=True,
)
print(test_emb)
print(f"  Embedding dimension : {len(test_emb)}")


[-0.01312256 -0.02563477 -0.00656128 ...  0.01660156  0.01525879
  0.01611328]
  Embedding dimension : 1024


In [40]:
help(embed_model.encode_document)

Help on method encode_document in module sentence_transformers.sentence_transformer.model:

encode_document(
    inputs: 'list[SingleInput] | SingleInput',
    prompt_name: 'str | None' = None,
    prompt: 'str | None' = None,
    batch_size: 'int' = 32,
    show_progress_bar: 'bool | None' = None,
    output_value: "Literal['sentence_embedding', 'token_embeddings'] | None" = 'sentence_embedding',
    precision: "Literal['float32', 'int8', 'uint8', 'binary', 'ubinary']" = 'float32',
    convert_to_numpy: 'bool' = True,
    convert_to_tensor: 'bool' = False,
    device: 'str | list[str | torch.device] | None' = None,
    normalize_embeddings: 'bool' = False,
    truncate_dim: 'int | None' = None,
    pool: "dict[Literal['input', 'output', 'processes'], Any] | None" = None,
    chunk_size: 'int | None' = None,
    **kwargs
) -> 'list[Tensor] | np.ndarray | Tensor | dict[str, Tensor] | list[dict[str, Tensor]]' method of sentence_transformers.sentence_transformer.model.SentenceTransformer 

In [41]:
# overriding functions to make their repeative calling simpler

def encode_query(query: str) -> List[float]:
    """Embed a query using the model's built-in 'query' prompt (no manual prefix needed)."""
    return embed_model.encode_query(
        query,
        normalize_embeddings=True,
    ).tolist()

def encode_document(texts: List[str], batch_size: int = 64) -> np.ndarray:
    """Embed document chunks using the model's built-in 'document' prompt."""
    return embed_model.encode_document(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

In [42]:
test = embed_model.encode_query("What is the meaning of word Granth?")
print(test)
print(f"  Embedding dimension : {len(test)}")

[ 0.01477051 -0.05493164 -0.00370789 ...  0.00218201 -0.06298828
 -0.01928711]
  Embedding dimension : 1024


In [47]:
test = embed_model.encode_document(["What is the meaning of word Granth?","What is the capital of India?"])
print(test)
print(f"  Embedding dimension : {len(test[0])}")

[[-0.00445557 -0.07128906 -0.0062561  ...  0.01745605 -0.05737305
  -0.01867676]
 [-0.03735352 -0.04150391 -0.0045166  ...  0.05200195  0.01464844
  -0.03564453]]
  Embedding dimension : 1024


In [48]:
!pip install langchain-chroma

In [49]:
chunks[0]

Document(metadata={'source': 'doc_0', 'doc_id': 0}, page_content='Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society')

In [50]:
chunk_texts = [c.page_content for c in chunks]
embeddings = embed_model.encode_document(chunk_texts)

In [51]:
len(chunk_texts)

901

In [54]:
embeddings[0]

array([ 0.05932617, -0.0480957 , -0.01135254, ..., -0.0402832 ,
       -0.01733398,  0.04296875], dtype=float32)

In [85]:
from langchain_chroma import Chroma
import chromadb

# client = chromadb.Client()

In [86]:
client = chromadb.PersistentClient(path="./my_local_db")

In [87]:
collection = client.get_or_create_collection(
    name="contexts",
    configuration={"hnsw": {"space": "cosine"}},
)

In [88]:
vector_store_from_client = Chroma(
    client=client,
    collection_name="contexts",
    embedding_function=embeddings,
)

In [89]:
chunks[0]

Document(metadata={'source': 'doc_0', 'doc_id': 0}, page_content='Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society')

In [90]:
# chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]
# chunk_metadatas = [
#     {
#         "source": c.metadata.get("source", ""),
#         "doc_id": str(c.metadata.get("doc_id", "")),
#     }
#     for c in chunks
# ]

chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]
chunk_metadatas = [chunks[i].metadata for i in range(len(chunks))]

In [91]:
chunk_metadatas[0]

{'source': 'doc_0', 'doc_id': 0}

In [92]:
print(f"✓ Computed {len(embeddings)} embeddings out of {len(chunks)}")

✓ Computed 901 embeddings out of 901


In [76]:
BATCH = 200
print("\nInserting into ChromaDB ...")
for start in range(0, len(chunks), BATCH):
    end = min(start + BATCH, len(chunks))
    collection.add(
        documents  = chunk_texts[start:end],
        embeddings = embeddings[start:end].tolist(),
        ids        = chunk_ids[start:end],
        metadatas  = chunk_metadatas[start:end],
    )
    print(f"  {end}/{len(chunks)} chunks indexed")

print(f"\n✓ ChromaDB ready — {collection.count()} documents indexed")


Inserting into ChromaDB ...
  200/901 chunks indexed
  400/901 chunks indexed
  600/901 chunks indexed
  800/901 chunks indexed
  901/901 chunks indexed

✓ ChromaDB ready — 901 documents indexed


In [77]:
from huggingface_hub import login; login()

In [78]:
# ── Load Gemma 3 4B from HuggingFace ────────────────────────────────────────

MODEL_ID = "google/gemma-3-4b-it" # gemma 3 is a gated model

# Optional 4-bit quantization — reduces VRAM from ~8 GB to ~4 GB
from transformers import BitsAndBytesConfig
quant_cfg = BitsAndBytesConfig(load_in_4bit=True)

print(f"Loading {MODEL_ID} ...")
_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quant_cfg,
)

llm_pipeline = pipeline(
    "text-generation",
    model=_model,
    tokenizer=_tokenizer,
)
# device_map="auto" may distribute across multiple devices; hf_device_map shows the layout
device_info = getattr(_model, "hf_device_map", str(next(_model.parameters()).device))
print(f"✓ {MODEL_ID} loaded  |  device map: {device_info}")

Loading google/gemma-3-4b-it ...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✓ google/gemma-3-4b-it loaded  |  device map: cuda:0


In [97]:
def generate(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate a response using Gemma 3 4B via HuggingFace Transformers."""
    outputs = llm_pipeline(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return outputs[0]["generated_text"][-1]["content"]


def naive_rag(query: str, n_results: int = 5) -> Dict:
    """Baseline RAG: retrieve top-k chunks, build context, generate answer."""
    print(type(vector_store_from_client))
    a = vector_store_from_client

    retriever = vector_store_from_client.as_retriever(
        search_type="similarity",
        search_kwargs={"k": n_results}
    )
    print("Hello")
    results = retriever.invoke(query)
    docs    = results["documents"][0]
    dists   = results["distances"][0]

    context = "\n\n---\n\n".join(docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":          query,
        "answer":         generate(prompt),
        "retrieved_docs": docs,
        "distances":      dists,
    }


# ── Quick test ──────────────────────────────────────────────────────────────
test_query   = dataset[0]["question"]
ground_truth = dataset[0]["answer"]

print(f"Query        : {test_query}")
print(f"Ground truth : {ground_truth}")
print("=" * 70)

result_naive = naive_rag(test_query)
print(f"\n[Naive RAG] Answer:\n{result_naive['answer']}")
print(f"\nTop retrieved chunk (distance={result_naive['distances'][0]:.4f}):")
print(result_naive["retrieved_docs"][0][:300] + "...")

Query        : Who found the answer to a search query collar george herbert essay?
Ground truth : Francisco Rogers found the answer to a search query collar george herbert essay.
<class 'langchain_chroma.vectorstores.Chroma'>


AttributeError: 'numpy.ndarray' object has no attribute 'embed_query'